# Manage Quota

Note: 
Quota check has to be enabled in Settings, e.g.
```toml
[quota]
mode = "check"
```

In [1]:
import itertools
import time
from collections.abc import Callable
from typing import TypeVar

import numpy as np
import requests as re

import geoengine as ge

In [2]:
R = TypeVar("R")


def wait(
    execute_fn: Callable[[], R],
    check_fn: Callable[[R], bool],
    *,
    message: str = "Waiting...",
    done_message: str = "✓ Done!",
    delay: float = 1,
) -> R:
    """Wait for a condition with a spinner animation."""

    def reprinter() -> Callable[[str], None]:
        max_length = 0

        def print_with_padding(output: str) -> None:
            nonlocal max_length
            max_length = max(max_length, len(output))
            padding = " " * max(max_length - len(output), 0)
            print(f"\r{output}{padding}", end="", flush=True)

        return print_with_padding

    spinner = itertools.cycle(["|", "/", "-", "\\"])
    reprint = reprinter()

    result = execute_fn()

    while not check_fn(result):
        output = f"{next(spinner)} {message}"
        reprint(output)

        time.sleep(delay)

        result = execute_fn()

    # Clear the line by padding with spaces
    reprint(done_message)

    return result

# Select a user

In [3]:
email = "foo@example.com"
password = "secret123"

# register the user, if it doesn't exist yet

assert re.post(
    "http://localhost:3030/api/user", json={"email": email, "password": password, "realName": "Foo Bar"}, timeout=60
).ok, "User registration failed"

## Initialize as user

In [4]:
ge.initialize("http://localhost:3030/api", credentials=(email, password))

user_id = ge.get_session().user_id

## Access own quota

In [5]:
ge.get_quota()

Quota(available=0, used=0)

## Try to run a query (fails, because quota is exhausted)

In [6]:
ports = ge.register_workflow(ge.workflow_builder.operators.OgrSource("ne_10m_ports"))

query = ge.QueryRectangle(
    ge.BoundingBox2D(-180.0, -90.0, 180.0, 90.0),
    ge.TimeInterval(np.datetime64("2014-04-01T12:00:00")),
)

try:
    df = ports.get_dataframe(query)
except ge.BadRequestException as e:
    print(e)

CreatingProcessorFailed: CreatingProcessorFailed: QuotaExhausted


## Initialize Geo Engine as Admin

In [7]:
ge.initialize("http://localhost:3030/api", ("admin@localhost", "adminadmin"))

## Access user quota

In [8]:
ge.get_quota(user_id)

Quota(available=0, used=0)

## Update user quota

In [9]:
ge.update_quota(user_id, 1000)

## Verify quota update worked

In [10]:
ge.get_quota(user_id)

Quota(available=1000, used=0)

# Go back to the regular user

In [11]:
ge.initialize("http://localhost:3030/api", credentials=(email, password))

ge.get_quota()

Quota(available=1000, used=0)

# Rerun the workflow, works now

In [12]:
df = ports.get_dataframe(query)


df.head()

,geometry,scalerank,featurecla,website,name,natlscale,start,end
0,POINT (-69.92356 12.4375),8,Port,www.rocargo.com/SanNicolas.html,Sint Nicolaas,5.0,NaT,NaT
1,POINT (-58.95141 -34.15333),8,Port,www.consejoportuario.com.ar,Campana,5.0,NaT,NaT
2,POINT (-59.00495 -34.09889),8,Port,www.consejoportuario.com.ar,Zarate,5.0,NaT,NaT
3,POINT (-62.10088 -38.89444),8,Port,NaN,Puerto Belgrano/Bahia Blanca,5.0,NaT,NaT
4,POINT (-62.30053 -38.78306),8,Port,NaN,Puerto Galvan/Bahia Blanca,5.0,NaT,NaT


## Verify that the used quota was recorded

In [13]:
wait(
    execute_fn=ge.get_quota,
    check_fn=lambda q: q.used != 0,
    message="Waiting for quota to be updated...",
    done_message="✓ Quota updated!",
)

✓ Quota updated!                    

Quota(available=999, used=1)

## Get Computation ID from dataframe

In [14]:
computation_id = df.attrs["computation_id"]

print(f"Computation `{computation_id}` used {ge.data_usage_for_computation(computation_id)} credits.")

Computation `1ea47062-7ff8-4db8-ad4a-c88fdd42a06d` used 1 credits.
